<a href="https://colab.research.google.com/github/e23189uop/Statistical-Learning-e23189/blob/main/assignment%2305_e23189/assignment05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Part A: Theoretical Fundamentals**

**1. Maximum Likelihood Estimator Bias and Bessel's Correction**

* **Derivation of MLE Bias:**
The Maximum Likelihood Estimator for the covariance matrix is given by $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n}\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T$.
To find its expectation, we can add and subtract the true population mean $\boldsymbol{\mu}$:

$$\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T = \sum_{i=1}^n \left( (\mathbf{X}_i - \boldsymbol{\mu}) - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \right) \left( (\mathbf{X}_i - \boldsymbol{\mu}) - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \right)^T$$



Expanding this quadratic form and taking the sum yields:

$$\sum_{i=1}^n (\mathbf{X}_i - \boldsymbol{\mu})(\mathbf{X}_i - \boldsymbol{\mu})^T - n(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})^T$$



Taking the expected value $\mathbb{E}[\cdot]$ of both terms:
The first term is $n \mathrm{Var}(\mathbf{X}_i) = n\boldsymbol{\Sigma}$.
The second term relies on the variance of the sample mean, which is $\frac{1}{n}\boldsymbol{\Sigma}$. Thus, $n \mathbb{E}[(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})^T] = n \left(\frac{1}{n}\boldsymbol{\Sigma}\right) = \boldsymbol{\Sigma}$.
Subtracting these gives:

$$\mathbb{E}\left[\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T\right] = n\boldsymbol{\Sigma} - \boldsymbol{\Sigma} = (n-1)\boldsymbol{\Sigma}$$



Dividing by $n$ completes the proof:

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{n-1}{n}\boldsymbol{\Sigma}$$


* **Bessel's Correction:**
Because the MLE underestimates the true covariance by a factor of $\frac{n-1}{n}$, we apply Bessel's correction by multiplying $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}$ by $\frac{n}{n-1}$:

$$\mathbf{S} = \frac{n}{n-1} \widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n-1} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T$$



Taking the expectation yields $\mathbb{E}[\mathbf{S}] = \frac{n}{n-1} \left( \frac{n-1}{n}\boldsymbol{\Sigma} \right) = \boldsymbol{\Sigma}$, proving it is an unbiased estimator.

**2. Decision Space in SHM**

* **Type I and Type II Errors:**
* **Type I Error ($\alpha$):** A "False Alarm." The diagnostic system triggers an alert indicating structural damage when the asset is actually perfectly healthy.
* **Type II Error ($\beta$):** A "Missed Detection." The diagnostic system fails to trigger an alert, assuming the structure is healthy when actual physical damage or drift has occurred.


* **Consequence of Ultra-Conservative Thresholds:**
Setting $\alpha = 0.0001$ drastically reduces the probability of a false alarm, but by the mathematical trade-off between error types, it heavily inflates $\beta$, thereby plummeting the statistical power ($1 - \beta$) of the test. Geometrically, this forces the "healthy operation" confidence ellipsoid to expand significantly in volume. It becomes so large that subtle, true structural anomalies fall within the expected boundary and go entirely undetected.

### **Part B: Theoretical Extension**

* **Slutsky's Theorem:**
If a sequence of random vectors $\mathbf{X}_n$ converges in distribution to a random vector $\mathbf{X}$ ($\mathbf{X}_n \xrightarrow{d} \mathbf{X}$), and another sequence of random matrices $\mathbf{Y}_n$ converges in probability to a constant matrix $\mathbf{C}$ ($\mathbf{Y}_n \xrightarrow{p} \mathbf{C}$), then $\mathbf{Y}_n \mathbf{X}_n \xrightarrow{d} \mathbf{C}\mathbf{X}$.
* **Operationalizing the Parametric Distribution:**
By the Weak Law of Large Numbers (WLLN), the unbiased sample covariance matrix converges to the population matrix: $\mathbf{S} \xrightarrow{p} \boldsymbol{\Sigma}$. By continuous mapping, $\mathbf{S}^{-1/2} \xrightarrow{p} \boldsymbol{\Sigma}^{-1/2}$.
From the Central Limit Theorem, $\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$.
Applying Slutsky's Theorem, substituting $\mathbf{S}$ for $\boldsymbol{\Sigma}$ maintains the asymptotic standard normal distribution structure, justifying the operational assumption that $\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left({\boldsymbol{\mu}}, \frac{1}{n}\mathbf{S}\right)$ for large $n$.

### **Part C: Numerical Verification**

Here is the Python implementation to verify first-moment homogeneity:


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment homogeneity
    via Wilks' Lambda and Bartlett's Chi-Square asymptotic transformation.
    """
    n, m = df.shape
    chunk_size = n // g_chunks
    global_mean = df.mean().values

    # Initialize Within-Chunk (W) and Between-Chunk (B) matrices
    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for i in range(g_chunks):
        chunk = df.iloc[i * chunk_size : (i + 1) * chunk_size]
        chunk_mean = chunk.mean().values

        # Calculate W
        diff_W = chunk.values - chunk_mean
        W += diff_W.T @ diff_W

        # Calculate B
        diff_B = (chunk_mean - global_mean).reshape(-1, 1)
        B += chunk_size * (diff_B @ diff_B.T)

    # Calculate Wilks' Lambda
    wilks_lambda = np.linalg.det(W) / np.linalg.det(B + W)

    # Calculate Bartlett's Chi-Square approximation
    chi2_calc = -(n - 1 - (m + g_chunks) / 2) * np.log(wilks_lambda)

    # Degrees of freedom and p-value
    df_chi2 = m * (g_chunks - 1)
    p_value = 1 - stats.chi2.cdf(chi2_calc, df_chi2)

    return {
        "Wilks_Lambda": wilks_lambda,
        "Bartlett_Chi2": chi2_calc,
        "p_value": p_value
    }



## **Geometric Subspace Optimization via PCA**

### **Part A: Theoretical Fundamentals**

**1. Coordinate Projections & Orthogonality**

* **Derivation of Transformed Covariance:**

$$\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[(\mathbf{P}^T \widetilde{\mathbf{X}}_i)(\mathbf{P}^T \widetilde{\mathbf{X}}_i)^T] = \mathbb{E}[\mathbf{P}^T \widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T \mathbf{P}]$$



Because the projection matrix $\mathbf{P}$ is deterministic, we can pull it out of the expectation:

$$\mathbf{P}^T \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T] \mathbf{P} = \mathbf{P}^T \boldsymbol{\Sigma} \mathbf{P}$$



Substitute the spectral decomposition $\boldsymbol{\Sigma} = \mathbf{P} \mathbf{\Lambda} \mathbf{P}^T$:

$$\mathbf{P}^T (\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) \mathbf{P}$$



Since $\mathbf{P}$ is orthogonal, $\mathbf{P}^T\mathbf{P} = \mathbf{I}$. This simplifies exactly to $\mathbf{\Lambda}$.
* **Geometric and Statistical Meaning:**
Because $\mathbf{\Lambda}$ is strictly diagonal, the off-diagonal elements are exactly zero. Statistically, this means the principal components $Z_{i,j}$ and $Z_{i,k}$ are completely uncorrelated. Geometrically, the new coordinate axes are strictly orthogonal to one another, rotating the data so that its variance aligns cleanly with the standard basis vectors.

**2. Variance Conservation**

* **Proof via Trace:**

$$\text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T)$$



Using the cyclic permutation property of the trace operator ($\text{tr}(\mathbf{A}\mathbf{B}\mathbf{C}) = \text{tr}(\mathbf{C}\mathbf{A}\mathbf{B})$):

$$\text{tr}(\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) = \text{tr}(\mathbf{P}^T \mathbf{P} \mathbf{\Lambda}) = \text{tr}(\mathbf{I} \mathbf{\Lambda}) = \text{tr}(\mathbf{\Lambda})$$



Since $\mathbf{\Lambda}$ is a diagonal matrix containing eigenvalues, $\text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j$. Total system energy is mathematically conserved.
* **Variance Ratios:**

$$\Phi(k) = \frac{\sum_{j=1}^k \lambda_j}{\sum_{j=1}^m \lambda_j} \quad \text{and} \quad \Psi(k) = \frac{\sum_{j=k+1}^m \lambda_j}{\sum_{j=1}^m \lambda_j} = 1 - \Phi(k)$$



**3. Vector Reconstruction**

* **Definitions:**
Reconstructed vector: $\widehat{\mathbf{x}}_i = \widehat{\mathbf{P}}_k \mathbf{z}_{i,k}$
Residual error vector: $\mathbf{e}_i = \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$
* **Pythagorean Proof:**
Because the subspaces defined by $\widehat{\mathbf{P}}_k$ and $\widehat{\mathbf{P}}_{m-k}$ are orthogonal, their dot product is zero.

$$\|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\widehat{\mathbf{P}}_k \mathbf{z}_{i,k} + \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}\|^2 = \|\widehat{\mathbf{P}}_k \mathbf{z}_{i,k}\|^2 + \|\widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2$$


* **$T^2$ vs $Q$ (SPE):**
* **Hotelling's $T^2$** tracks the major operational subspace. An environmental load anomaly (like high wind loads) shifts the structure within its normal expected physical pathways, spiking the $T^2$ statistic.
* **The $Q$ Statistic** tracks the residual "noise" space. An internal structural fracture breaks the physical correlation between sensors, violating the historical covariance matrix and causing the unexpected error vector to surge, thus spiking the $Q$ statistic.



---

## **Latent Subspace Decomposition via Factor Analysis**

### **Part A: Theoretical Fundamentals**

**1. Generative Model & Commonalities**

* **Fundamental Equation Proof:**

$$\mathbf{R} = \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[(\boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i)(\boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i)^T]$$



Expanding the terms:

$$\mathbf{R} = \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\boldsymbol{\epsilon}_i^T] + \mathbb{E}[\boldsymbol{\epsilon}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T]$$



Since factors and errors are independent ($\mathbb{E}[\boldsymbol{\epsilon}_i \mathbf{F}_i^T] = \mathbf{0}$) and factors are standard orthogonal ($\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] = \mathbf{I}$), this collapses to:

$$\mathbf{R} = \boldsymbol{\Lambda}\mathbf{I}\boldsymbol{\Lambda}^T + \mathbf{0} + \mathbf{0} + \boldsymbol{\Psi} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$$


* **Communality vs Uniqueness:**
* **Communality ($h_j^2$):** Represents the proportion of variance in sensor $j$ driven by the shared, global physical structures (the latent factors).
* **Uniqueness ($\varphi_j^2$):** Represents the isolated variance specific only to sensor $j$, typically capturing localized measurement noise or specific instrument drift.



**2. Factor Rotation**

* **PCA vs Varimax:** PCA forces an artificial hierarchy where the first component maximizes as much mixed variance as possible, leading to dense, hard-to-read loading vectors. Varimax rotation deliberately redistributes this variance to maximize "simple structure," pushing loadings toward 0 or 1, making it clear which sensors align with which underlying physical mechanisms.
* **Orthogonal Rotation Invariance:** If we apply rotation matrix $\mathbf{T}$ where $\mathbf{T}\mathbf{T}^T = \mathbf{I}$, the new loadings are $\widetilde{\boldsymbol{\Lambda}} = \boldsymbol{\Lambda}\mathbf{T}$. The global covariance approximation remains unchanged: $(\boldsymbol{\Lambda}\mathbf{T})(\boldsymbol{\Lambda}\mathbf{T})^T + \boldsymbol{\Psi} = \boldsymbol{\Lambda}(\mathbf{T}\mathbf{T}^T)\boldsymbol{\Lambda}^T + \boldsymbol{\Psi} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$. The row sums of squares (communality) are preserved.

**3. Thomson’s Regression Method**

* **Joint Covariance:**
The cross-covariance between $\mathbf{Z}_i$ and $\mathbf{F}_i$ is $\mathbb{E}[\mathbf{Z}_i\mathbf{F}_i^T] = \mathbb{E}[(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)\mathbf{F}_i^T] = \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] + \mathbf{0} = \boldsymbol{\Lambda}$. Therefore, the joint partitioned matrix $\boldsymbol{\Sigma}_{YY}$ is:

$$\boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbb{E}[\mathbf{Z}\mathbf{Z}^T] & \mathbb{E}[\mathbf{Z}\mathbf{F}^T] \\ \mathbb{E}[\mathbf{F}\mathbf{Z}^T] & \mathbb{E}[\mathbf{F}\mathbf{F}^T] \end{bmatrix} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I}_{k \times k} \end{bmatrix}$$


* **Thomson's Estimator:**
Applying the conditional mean projection formula:

$$\mathbb{E}[\mathbf{F}_i | \mathbf{z}_i] = \mathbf{0} + \boldsymbol{\Lambda}^T \mathbf{R}^{-1} (\mathbf{z}_i - \mathbf{0}) = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$$



---

## **Subspace Diagnostics & Feature De-correlation Analysis**

### **Question 1: The Total Variance Illusion**

1. **Physical Nature of Sensor 4:** A Uniqueness value ($\varphi^2$) exceeding 98% mathematically proves that Sensor 4 is entirely disconnected from the physical dynamics of the asset. It is essentially outputting pure, uncorroborated electrical noise or static.
2. **PCA's Vulnerability:** PCA is entirely blind to the *source* of variance; it only chases magnitude. Because Sensor 4 has an artificially massive variance scale ($\sigma^2 \approx 2.0$), the PCA engine maximizes its objective function by dedicating a massive chunk of its primary eigenvectors simply to model this localized noise.
3. **The Engineering Risk:** If an operator relies solely on PCA, a short-circuiting sensor (generating massive variance) will warp the principal subspace. The operator might misinterpret this massive variance jump as a catastrophic, plant-wide structural failure, triggering an expensive false alarm (Type I error) to shut down the plant, when all that was needed was a $50 sensor replacement.

### **Question 2: Decoupling Structural Loading via Rotation**

1. **Varimax Relaxation:** Traditional PCA forces eigenvectors to be orthogonal while strictly maximizing variance in descending order. Varimax rotation abandons the strict descending variance hierarchy. By iteratively rotating the axes in the latent space, it drives intermediate loading values toward zero or one, cleanly decoupling the mixed signals into distinct, isolated clusters.
2. **Troubleshooting Advantage:** During a crisis, operators need immediate localization. An unrotated PCA matrix presents a blend of variables, making it unclear what exactly broke. A rotated FA heatmap assigns specific sensors explicitly to specific underlying factors. If "Factor 2" suddenly spikes, operators immediately know to inspect `Sensor_3` and its associated structural subsystem, skipping unnecessary diagnostics.

### **Question 3: Determining Subspace Truncation ($k$)**

1. **Behavior of the $Q$ Statistic:** As $k$ moves from 1 to 2, the Mean $Q$ statistic drops precipitously because a massive portion of genuine physical variance is shifted from the residual error pool into the explained subspace. Moving from $k=2$ to $k=3$, the curve flattens out entirely (forming an "elbow").
2. **Identifying the Dimensionality:** The sharp drop followed by a flat line mathematically indicates that the asset is driven by exactly two genuine physical processes. Beyond $k=2$, there is no more structural information left to extract.
3. **The Overfitting Penalty:** Forcing a cutoff of $k=3$ pushes pure, uncorroborated measurement noise out of the residual term and artificially embeds it into your "clean" structural monitoring threshold, muddying your diagnostic metrics and increasing false-alarm susceptibility.

### **Question 4: Operational Trade-offs in System Health Monitoring**

* **Comparison:** * *PCA Strategy:* Monitors a fused state of variance. It is sensitive to broad system-level shifts but highly vulnerable to being dragged off-center by heavy localized noise.
* *FA Strategy:* Methodically strips out uniqueness/noise, projecting only the cross-correlated, shared structural variance into the latent scores.


* **Robustness Conclusion:** The **FA Strategy** is vastly more robust. If a single sensor loses calibration or shorts out, its variance explodes, but FA's mathematical architecture correctly identifies that this variance is uncorrelated with the rest of the grid. It automatically dumps this surge into that specific sensor's **Uniqueness ($\varphi^2$)** parameter, completely protecting the shared Latent Factor Scores from contamination.